# Stage 0.2: PatternReframe preprocessing

Source: `reframe_thoughts_dataset_parlai_xxx.txt` .
Each of the three splits (`train`, `valid`, `test`) is a JSON-lines file with one record per line.

**Output:** `processed/patternreframe_{train,val,test}.csv`.

**Pipeline:**

1. Parse the JSON-lines files.
2. Flatten each record into a tabular row.
3. Flatten the soft-label `marked_patterns` distribution into one column per pattern.
4. Extract the writer-intended single label.
5. Apply a minimum-length filter.
6. Normalise whitespace and unicode escapes.
7. Output the processed CSV.

## Step 1: Setup

In [2]:
from __future__ import annotations

import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [3]:
INPUT_DIR = Path("./datasets/")
OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILES = {
    "train": INPUT_DIR / "reframe_thoughts_dataset_parlai_train.txt",
    "val":   INPUT_DIR / "reframe_thoughts_dataset_parlai_valid.txt",
    "test":  INPUT_DIR / "reframe_thoughts_dataset_parlai_test.txt",
}

# Expected sizes reported by Maddela et al. (2023). Used as a sanity check.
EXPECTED_SIZES = {"train": 1_920, "val": 961, "test": 6_807}

MIN_THOUGHT_LENGTH = 10  # characters; per Stage 0.2 step 5 in the master plan
MAX_REFRAMES = 3         # up to 3 reframes per thought in the dataset

## Step 2: Category inventory

Two related vocabularies are used in the raw records and must be kept apart.

The writer-intended single label is stored in the field `pattern` and takes one of ten canonical values. *Emotional reasoning* never appears as a writer-intended label because it was not used as a conditioning prompt in Task 1 of the data collection pipeline.

The soft-label vote distribution is stored in the field `marked_patterns` and has twelve keys: the same ten patterns plus *Emotional reasoning* plus *None*. The order of keys is fixed below so that the output columns are reproducible across splits. A short-form suffix is attached to each column name to keep the CSV header readable.

In [4]:
# Canonical order of the 10 writer-intended categories.
CANONICAL_PATTERNS = [
    "Catastrophizing",
    "Discounting the positive",
    "Labeling and mislabeling",
    "Mental filtering",
    "Jumping to conclusions: mind reading",
    "Jumping to conclusions: Fortune-telling",
    "Overgeneralization",
    "Personalization",
    "Black-and-white or polarized thinking / All or nothing thinking",
    "Should statements",
]

# The 12-key vocabulary used inside marked_patterns. Order preserved for
# reproducible column layout across train/val/test.
MARKED_PATTERN_KEYS = CANONICAL_PATTERNS + [
    "Emotional reasoning",
    "None",
]

# Short suffix used when flattening marked_patterns into columns.
SHORT_SUFFIX = {
    "Catastrophizing": "catastrophizing",
    "Discounting the positive": "discounting_positive",
    "Labeling and mislabeling": "labeling",
    "Mental filtering": "mental_filtering",
    "Jumping to conclusions: mind reading": "mind_reading",
    "Jumping to conclusions: Fortune-telling": "fortune_telling",
    "Overgeneralization": "overgeneralization",
    "Personalization": "personalization",
    "Black-and-white or polarized thinking / All or nothing thinking": "black_and_white",
    "Should statements": "should_statements",
    "Emotional reasoning": "emotional_reasoning",
    "None": "none",
}

## Step 3: Helper functions

The first helper normalises whitespace and unicode escapes in a text field. Multiple whitespace characters are collapsed to a single space, leading and trailing whitespace are stripped, and the text is put through NFKC normalisation so that escaped sequences such as `\u2019` (typographic apostrophe) are resolved consistently.

The second flattens the `marked_patterns` object into a fixed vector of vote counts in the canonical key order. Raw values in the dataset are stored as strings and are cast to `int`.

The third normalises `marked_strategy` keys, which carry trailing spaces in the raw dataset (for example `"Growth Mindset "` instead of `"Growth Mindset"`). A clean dictionary is produced.

The fourth serialises the per-reframe strategy annotations to a JSON string. This preserves the nested structure in a single column so that it can be read back with `json.loads` later in Stage 7.

In [5]:
_WHITESPACE_RE = re.compile(r"\s+")

def clean_text(text: str | None) -> str:
    """Normalise unicode, collapse whitespace, strip ends."""
    if text is None:
        return ""
    t = unicodedata.normalize("NFKC", str(text))
    t = _WHITESPACE_RE.sub(" ", t).strip()
    return t

def flatten_marked_patterns(marked: dict) -> dict:
    """Return a dict keyed by short column names with integer vote counts.

    Missing keys (shouldn't happen on this dataset, but handled defensively)
    are filled with 0.
    """
    out: dict = {}
    for key in MARKED_PATTERN_KEYS:
        raw = marked.get(key, 0)
        try:
            out[f"marked_{SHORT_SUFFIX[key]}"] = int(raw)
        except (TypeError, ValueError):
            out[f"marked_{SHORT_SUFFIX[key]}"] = 0
    return out

def normalise_strategy_keys(marked_strategy: dict) -> dict:
    """Strip trailing whitespace from the strategy keys and cast values to int."""
    return {k.strip(): int(v) for k, v in marked_strategy.items()}

def serialise_strategies(reframes: list[dict]) -> str:
    """Pack the per-reframe strategy annotations into a JSON string.

    Output schema (list, one entry per reframe):
        [{"marked_strategy": {...}, "original_strategy": [...]}, ...]
    """
    packed = []
    for rf in reframes:
        packed.append({
            "marked_strategy": normalise_strategy_keys(rf.get("marked_strategy", {})),
            "original_strategy": list(rf.get("original_strategy", [])),
        })
    return json.dumps(packed, ensure_ascii=False)

## Step 4: Parsing and flattening a single split

One row is produced per thought. The up-to-three reframes are exploded into three dedicated columns `reframe_1`, `reframe_2`, `reframe_3`. Records with fewer reframes (52 records with 1 and 407 with 2 in the train split, per the dataset analysis) leave the unused columns as empty strings. An `n_reframes` column is added for convenience so that the true count is not lost.

Length filtering drops any thought with fewer than `MIN_THOUGHT_LENGTH` characters after cleaning. No upper bound is imposed because the average thought is roughly 89 characters long (Maddela et al., 2023).

In [6]:
def load_split(path: Path) -> list[dict]:
    """Read a ParlAI .txt file and return a list of parsed JSON records."""
    records: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Malformed JSON at {path.name}:{lineno}") from exc
    return records

def build_dataframe(records: list[dict]) -> tuple[pd.DataFrame, dict]:
    """Flatten records into a DataFrame and return a small stats dict."""
    rows = []
    dropped_short = 0
    reframe_count_hist: Counter = Counter()
    unknown_patterns: Counter = Counter()

    for rec in records:
        thought_clean = clean_text(rec.get("thought", ""))
        if len(thought_clean) < MIN_THOUGHT_LENGTH:
            dropped_short += 1
            continue

        pattern = rec.get("pattern", "")
        if pattern not in CANONICAL_PATTERNS:
            unknown_patterns[pattern] += 1

        reframes = rec.get("reframes", []) or []
        reframe_count_hist[len(reframes)] += 1

        reframe_cols = {f"reframe_{i + 1}": "" for i in range(MAX_REFRAMES)}
        for i, rf in enumerate(reframes[:MAX_REFRAMES]):
            reframe_cols[f"reframe_{i + 1}"] = clean_text(rf.get("reframe", ""))

        row = {
            "persona": clean_text(rec.get("persona", "")),
            "pattern": pattern,
            "pattern_def": clean_text(rec.get("pattern_def", "")),
            "thought": thought_clean,
            "thought_length": len(thought_clean),
            "n_reframes": len(reframes),
            **reframe_cols,
            **flatten_marked_patterns(rec.get("marked_patterns", {})),
            "strategies_per_reframe": serialise_strategies(reframes),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # Enforce a stable column order.
    column_order = (
        ["persona", "pattern", "pattern_def", "thought", "thought_length", "n_reframes"]
        + [f"reframe_{i + 1}" for i in range(MAX_REFRAMES)]
        + [f"marked_{SHORT_SUFFIX[k]}" for k in MARKED_PATTERN_KEYS]
        + ["strategies_per_reframe"]
    )
    df = df.reindex(columns=column_order)

    stats = {
        "input_records": len(records),
        "output_rows": len(df),
        "dropped_short": dropped_short,
        "reframe_count_hist": dict(reframe_count_hist),
        "unknown_patterns": dict(unknown_patterns),
    }
    return df, stats

## Step 5: Run on every available split

Each split is processed independently.

In [7]:
split_frames: dict[str, pd.DataFrame] = {}
split_stats: dict[str, dict] = {}

for split_name, src_path in SPLIT_FILES.items():
    records = load_split(src_path)
    expected = EXPECTED_SIZES.get(split_name)
    if expected is not None and len(records) != expected:
        print(
            f"[WARN] {split_name}: {len(records)} records parsed, "
            f"but Maddela et al. report {expected}."
        )

    df, stats = build_dataframe(records)
    out_path = OUTPUT_DIR / f"patternreframe_{split_name}.csv"
    df.to_csv(out_path, index=False, encoding="utf-8")

    split_frames[split_name] = df
    split_stats[split_name] = stats

    print(
        f"[OK]   {split_name}: {stats['input_records']} parsed, "
        f"{stats['output_rows']} written, {stats['dropped_short']} dropped (<{MIN_THOUGHT_LENGTH} chars). "
        f"-> {out_path}"
    )

[OK]   train: 1920 parsed, 1920 written, 0 dropped (<10 chars). -> processed/patternreframe_train.csv
[OK]   val: 961 parsed, 961 written, 0 dropped (<10 chars). -> processed/patternreframe_val.csv
[OK]   test: 6807 parsed, 6807 written, 0 dropped (<10 chars). -> processed/patternreframe_test.csv


## Step  6: Sanity checks

1. the distribution of the writer-intended label in each split (expected to be roughly uniform, 9.5% – 10.6% per category);
2. the reframe-count distribution per split;
3. basic length statistics for the `thought` field;
4. a preview of the first flattened row to confirm that the column layout matches Stage 0.2 step 7 of the master plan.

In [8]:
# 6.1 Label distribution per split.
for split_name, df in split_frames.items():
    counts = df["pattern"].value_counts().reindex(CANONICAL_PATTERNS, fill_value=0)
    share = (counts / counts.sum() * 100).round(2)
    dist = pd.DataFrame({"count": counts, "share_%": share})
    print(f"=== {split_name} — writer-intended label distribution ===")
    print(dist)
    print()

=== train — writer-intended label distribution ===
                                                                 count  share_%
pattern                                                                        
Catastrophizing                                                    200    10.42
Discounting the positive                                           194    10.10
Labeling and mislabeling                                           196    10.21
Mental filtering                                                   185     9.64
Jumping to conclusions: mind reading                               197    10.26
Jumping to conclusions: Fortune-telling                            199    10.36
Overgeneralization                                                 197    10.26
Personalization                                                    185     9.64
Black-and-white or polarized thinking / All or nothing thinking    188     9.79
Should statements                                                  17

In [9]:
# 6.2 Reframe count distribution per split.
for split_name, stats in split_stats.items():
    print(f"{split_name}: reframes-per-thought =", dict(sorted(stats["reframe_count_hist"].items())))

train: reframes-per-thought = {1: 52, 2: 407, 3: 1461}
val: reframes-per-thought = {1: 37, 2: 186, 3: 738}
test: reframes-per-thought = {1: 221, 2: 1344, 3: 5242}


In [10]:
# 6.3 Length statistics for the cleaned thought field.
for split_name, df in split_frames.items():
    desc = df["thought_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    print(f"=== {split_name} — thought_length (characters) ===")
    print(desc.round(1))
    print()

=== train — thought_length (characters) ===
count    1920.0
mean       90.1
std        28.8
min        29.0
25%        70.0
50%        85.5
75%       105.0
95%       141.0
99%       182.0
max       267.0
Name: thought_length, dtype: float64

=== val — thought_length (characters) ===
count    961.0
mean      90.0
std       28.4
min       33.0
25%       69.0
50%       86.0
75%      105.0
95%      145.0
99%      181.4
max      215.0
Name: thought_length, dtype: float64

=== test — thought_length (characters) ===
count    6807.0
mean       88.8
std        28.3
min        25.0
25%        69.0
50%        84.0
75%       103.0
95%       141.0
99%       177.0
max       260.0
Name: thought_length, dtype: float64



In [11]:
# 6.4 Preview of the first row from the first available split.
if split_frames:
    any_split = next(iter(split_frames))
    preview = split_frames[any_split].head(1).T
    preview.columns = [f"{any_split}[0]"]
    display(preview)

,train[0]
persona,"i have a pencil thin mustache . i have six tattoos , all of cats . my mother is a ballet dancer . i am from baltimor..."
pattern,Catastrophizing
pattern_def,"Catastrophizing: Giving greater weight to the worst possible outcome, however unlikely, or experiencing a situation ..."
thought,I like my cats. I think one day they will plot against me and eat me in my sleep.
thought_length,81
n_reframes,3
reframe_1,"I like my cats. I have a fun imagination, but I know it's not likely they are plotting against me to eat me."
reframe_2,I like my cats. I’ll get used to living with them and will be less afraid as time goes on.
reframe_3,I like my cats so I will make sure to give them all the exercise and playtime that they need.
marked_catastrophizing,4


## Output

The three CSV files written to `processed/`. The schema is fixed to the following columns, in order:

- `persona` - character description from *PERSONA-CHAT* (cleaned);
- `pattern` - writer-intended category label (ten values; single-label target for classification);
- `pattern_def` - definition of the pattern shown to the crowdworker during data collection;
- `thought` - the unhelpful thought, cleaned;
- `thought_length` - character length of the cleaned thought;
- `n_reframes` - number of reframes present for this thought (1, 2, or 3);
- `reframe_1`, `reframe_2`, `reframe_3` - reframe strings; empty when `n_reframes` is below the column index;
- `marked_<short>` - twelve soft-label vote columns (one per pattern plus *Emotional reasoning* plus *None*), integer-valued;
- `strategies_per_reframe` - JSON string encoding the list of `{marked_strategy, original_strategy}` dicts, one per reframe.

Downstream stages consume these files as follows. Stage 1 (baselines) and Stage 2 (full-text transformer) use `thought` as input and `pattern` as the target. Later soft-label experiments can switch to the `marked_*` columns instead of the single label. Stage 4 (multilingual/Croatian) ingests the same tables after translation. Stage 7 (reframing) reads `thought` and the three `reframe_*` columns as reference outputs, and reads `strategies_per_reframe` when prompt conditioning on strategies is evaluated.